In [2]:
!python prepare_data.py

Loading data from: ./
Loaded 13398 parallel sentence pairs.
Train size: 10718
Test size: 2680
Saving the dataset (1/1 shards): 100%|█| 10718/10718 [00:00<00:00, 376472.24 exa
Saving the dataset (1/1 shards): 100%|█| 2680/2680 [00:00<00:00, 357461.51 examp
Saved dataset to: ./hf_dataset


In [1]:
!pip install -r requirements.txt
!pip install git+https://github.com/VarunGumma/IndicTransToolkit.git

  Cloning https://github.com/VarunGumma/IndicTransToolkit.git to /tmp/pip-install-2k9vtx9z/indictranstoolkit_00f1fcefb4274d62acab950bd3e525c5
  Running command git clone --filter=blob:none --quiet https://github.com/VarunGumma/IndicTransToolkit.git /tmp/pip-install-2k9vtx9z/indictranstoolkit_00f1fcefb4274d62acab950bd3e525c5

  Resolved https://github.com/VarunGumma/IndicTransToolkit.git to commit 3efb8418d0721b4ce267c2b3586899d313191357
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/VarunGumma/IndicTransToolkit.git to /tmp/pip-req-build-ovsq9203
  Running command git clone --filter=blob:none --quiet https://github.com/VarunGumma/IndicTransToolkit.git /tmp/pip-req-build-ovsq9203

  Resolved https://github.com/VarunGumma/IndicTransToolkit.git to commit 3efb8418d0721b4ce267c2b3586899d313191357
  Installing build dependencies ... done
  Getting requirements to build wheel ..

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Token has not been saved to git credential helper.


In [4]:
!python train_indictrans2.py

Loading tokenizer and model from ai4bharat/indictrans2-indic-indic-dist-320M...
Tokenizing dataset...
Parameter 'function'=<function main.<locals>.preprocess_function at 0x7f648721fba0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
Map: 100%|█████████████████████████| 2680/2680 [00:00<00:00, 3865.82 examples/s]
/teamspace/studios/this_studio/Exploratory/bengali_hindi/train_indictrans2.py:96: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Starting training In

In [1]:
!python train_nllb.py

tokenizer_config.json: 100%|███████████████████| 564/564 [00:00<00:00, 5.67MB/s]
sentencepiece.bpe.model: 100%|█████████████| 4.85M/4.85M [00:00<00:00, 22.5MB/s]
tokenizer.json: 100%|██████████████████████| 17.3M/17.3M [00:00<00:00, 83.9MB/s]
special_tokens_map.json: 3.55kB [00:00, 35.9MB/s]
config.json: 100%|█████████████████████████████| 846/846 [00:00<00:00, 12.1MB/s]
pytorch_model.bin: 100%|████████████████████| 2.46G/2.46G [00:04<00:00, 558MB/s]
model.safetensors:   8%|█▋                   | 199M/2.46G [00:01<00:04, 525MB/s]
generation_config.json: 100%|██████████████████| 189/189 [00:00<00:00, 3.18MB/s]
Tokenizing dataset...

model.safetensors:  19%|███▉                 | 467M/2.46G [00:02<00:04, 468MB/s]
Map: 100%|██████████████████████| 10718/10718 [00:00<00:00, 25133.51 examples/s]

Map: 100%|████████████████████████| 2680/2680 [00:00<00:00, 33290.99 examples/s]
/teamspace/studios/this_studio/Exploratory/bengali_hindi/train_nllb.py:71: FutureWarning: `tokenizer` is deprecated 

In [ ]:
from comet import download_model, load_from_checkpoint

def combine_predictions_with_qe(sources, nllb_preds, indic_preds):
    print("\n--- Combining outputs using COMET-Kiwi (Quality Estimation) ---")
    
    # Download and load the reference-free QE model
    print("Loading COMET-Kiwi model...")
    model_path = download_model("Unbabel/wmt22-cometkiwi-da")
    qe_model = load_from_checkpoint(model_path)
    
    # Prepare data format for COMET
    nllb_data = [{"src": src, "mt": mt} for src, mt in zip(sources, nllb_preds)]
    indic_data = [{"src": src, "mt": mt} for src, mt in zip(sources, indic_preds)]
    
    # Score both sets of predictions (this does NOT use the human references)
    print("Scoring NLLB predictions...")
    nllb_results = qe_model.predict(nllb_data, batch_size=8, gpus=1)
    
    print("Scoring IndicTrans2 predictions...")
    indic_results = qe_model.predict(indic_data, batch_size=8, gpus=1)
    
    combined_preds = []
    nllb_wins = 0
    indic_wins = 0
    
    # Iterate through every sentence and pick the highest-scoring translation
    for i in range(len(sources)):
        if nllb_results.scores[i] > indic_results.scores[i]:
            combined_preds.append(nllb_preds[i])
            nllb_wins += 1
        else:
            combined_preds.append(indic_preds[i])
            indic_wins += 1
            
    print(f"\nCombination complete!")
    print(f"NLLB chosen {nllb_wins} times.")
    print(f"IndicTrans2 chosen {indic_wins} times.")
    
    return combined_preds

In [3]:
! python evaluate_models.py

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution

--- Loading model from nllb_finetuned_bn_hi/final_model ---
Generating predictions...
100%|█████████████████████████████████████████| 158/158 [01:01<00:00,  2.58it/s]

Evaluating performance for NLLB-600M...
[nltk_data] Downloading package wordnet to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /t